# Multi-label classification

Each crab has two attributes, its colour (blue or orange) and its sex (male or female). The labels are learned from examples. The knowledge base adds that the two labels of the same attribute exclude each other. After training, the model is **queried** with formulas that were not in the knowledge base.

In [ ]:
import numpy as np
import pandas as pd
import torch
import sltn
from sltn import Signature, Interpretation, KB
from sltn.interpretation.operators import AggregPMeanError

torch.manual_seed(0)
np.random.seed(0)
device = "cpu"

## Data

The Leptograpsus crabs dataset: 200 crabs, 5 measurements each (frontal lobe, rear width, carapace length and width, body depth). The data are shuffled, and 160 crabs are kept for training and 40 for testing. Colour is 0 = blue, 1 = orange; sex is 0 = male, 1 = female.

In [ ]:
df = pd.read_csv("../datasets/crabs.dat", sep=" ", skipinitialspace=True).sample(frac=1, random_state=0)
features = torch.tensor(df[["FL", "RW", "CL", "CW", "BD"]].to_numpy()).float()
color = torch.tensor((df["sp"] == "O").to_numpy()).long()
sex = torch.tensor((df["sex"] == "F").to_numpy()).long()

x_train, x_test = features[:160].to(device), features[160:].to(device)
color_train, color_test = color[:160].to(device), color[160:].to(device)
sex_train, sex_test = sex[:160].to(device), sex[160:].to(device)
print(x_train.shape, x_test.shape)

## 1 — Signature

- sorts `Item` (a crab) and `Label`;
- a function `f: Item → Label`, the network: it maps a crab to a vector of four truth values (blue, orange, male, female);
- a predicate `P(v, l)`: the truth value of label `l` in the vector `v`;
- four constants for the labels, `L_blue`, `L_orange`, `L_male`, `L_female`;
- a variable for all the crabs, `x`, and one for the crabs with each label.

`P(f(x), L_blue)` reads "x is blue". Writing the labels as constants, rather than four predicates, keeps one network and one predicate for all of them.

In [ ]:
sig = Signature("crabs_clf")
sig.sort("Item")
sig.sort("Label")
sig.function("f", ["Item"], "Label")
sig.predicate("P", ["Label", "Label"])
for name in ["L_blue", "L_orange", "L_male", "L_female"]:
    sig.constant(name, "Label")
for name in ["x", "x_blue", "x_orange", "x_male", "x_female"]:
    sig.variable(name, "Item")
print(sig)

## 2 — Knowledge base and queries

Four clauses give the labels, one per label:

```
∀ x_blue: P(f(x_blue), L_blue)
```

Two clauses say that the labels of one attribute exclude each other, for every crab:

```
∀ x: ¬(P(f(x), L_blue) & P(f(x), L_orange))
∀ x: ¬(P(f(x), L_male) & P(f(x), L_female))
```

Nothing relates colour to sex. After training, three formulas are evaluated that are not in the knowledge base:

- φ1 `∀ x: P(f(x), L_blue) -> ¬P(f(x), L_orange)`: "blue crabs are not orange", a consequence of the exclusion rule;
- φ2 `∀ x: P(f(x), L_blue) -> P(f(x), L_orange)`: "blue crabs are orange", contradicted by it;
- φ3 `∀ x: P(f(x), L_blue) -> P(f(x), L_male)`: "blue crabs are male", about which the knowledge base says nothing.

In [ ]:
kb = KB([
    sig.parse("∀ x_blue: P(f(x_blue), L_blue)"),
    sig.parse("∀ x_orange: P(f(x_orange), L_orange)"),
    sig.parse("∀ x_male: P(f(x_male), L_male)"),
    sig.parse("∀ x_female: P(f(x_female), L_female)"),
    sig.parse("∀ x: ¬(P(f(x), L_blue) & P(f(x), L_orange))"),
    sig.parse("∀ x: ¬(P(f(x), L_male) & P(f(x), L_female))"),
])
queries = {
    "φ1 blue -> not orange": sig.parse("∀ x: (P(f(x), L_blue) -> ¬P(f(x), L_orange))"),
    "φ2 blue -> orange": sig.parse("∀ x: (P(f(x), L_blue) -> P(f(x), L_orange))"),
    "φ3 blue -> male": sig.parse("∀ x: (P(f(x), L_blue) -> P(f(x), L_male))"),
}

## 3 — Model

An MLP from the 5 measurements to 4 values, each through its own sigmoid. The exclusion within each attribute is left to the knowledge base.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(5, 16), torch.nn.ELU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(16, 16), torch.nn.ELU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(16, 8), torch.nn.ELU(),
            torch.nn.Linear(8, 4), torch.nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## 4 — Interpretation

`f` is the network, the label constants are one-hot vectors, and `P(v, l)` is the dot product `v · l`, which picks the component of `v` selected by `l`.

In [ ]:
interp = Interpretation(sig).to(device)
interp["Item"] = sltn.Type("Item", shape=(5,))
interp["Label"] = sltn.Type("Label", shape=(4,))
interp["f"] = MLP()
interp["P"] = lambda v, l: torch.sum(v * l, dim=-1, keepdim=True)
for k, name in enumerate(["L_blue", "L_orange", "L_male", "L_female"]):
    interp[name] = torch.eye(4)[k]
print(interp)

## 5 — Grounding the variables, and metrics

**Accuracy** is per label: each of the four values thresholded at 0.5 and compared with the truth, averaged over labels and crabs (1 − Hamming loss).

The **queries** are evaluated with a stricter `∀`, pMeanError with p = 5, closer to the minimum: a query should be true for (almost) every crab, not on average (Tutorial 3). `with_updates()` gives a copy of the interpretation that shares the groundings, so the change does not affect training.

In [ ]:
def ground(x, color, sex):
    interp["x"] = x
    interp["x_blue"], interp["x_orange"] = x[color == 0], x[color == 1]
    interp["x_male"], interp["x_female"] = x[sex == 0], x[sex == 1]


def accuracy(x, color, sex):
    interp.eval()
    with torch.no_grad():
        interp["x"] = x
        predicted = torch.cat([interp(f"P(f(x), {L})").value for L in ["L_blue", "L_orange", "L_male", "L_female"]], dim=1) > 0.5
    truth = torch.stack([color == 0, color == 1, sex == 0, sex == 1], dim=1)
    return (predicted == truth).float().mean().item()


strict = interp.with_updates()
strict.logic["∀"] = AggregPMeanError(p=5)


def query(phi, x):
    interp.eval()
    with torch.no_grad():
        strict["x"] = x
        return strict(phi).value.item()

## 6 — Training

The loss is $1 - \mathrm{SatAgg}(\mathcal{K})$, on minibatches of 64 crabs.

In [ ]:
optimizer = torch.optim.Adam(interp.parameters(), lr=0.001)

for epoch in range(1000):
    interp.train()
    for idx in torch.randperm(len(x_train)).split(64):
        ground(x_train[idx], color_train[idx], sex_train[idx])
        optimizer.zero_grad()
        loss = 1 - interp(kb).value
        loss.backward()
        optimizer.step()

    if epoch % 200 == 0 or epoch == 999:
        interp.eval()
        with torch.no_grad():
            ground(x_train, color_train, sex_train)
            sat = interp(kb).value.item()
        print(f"epoch {epoch:4d} | sat {sat:.3f} | acc train {accuracy(x_train, color_train, sex_train):.3f} "
              f"test {accuracy(x_test, color_test, sex_test):.3f}")

## 7 — Querying the trained model

The three queries on the 40 test crabs:

In [ ]:
for name, phi in queries.items():
    print(f"{name:24} {query(phi, x_test):.3f}")

φ1 is close to true and φ2 close to false: both follow from the exclusion rule, which the model learned. φ3 is also low. The knowledge base says nothing about it, but in the data about half of the blue crabs are female, and the strict ∀ (p = 5) is pulled down by those counterexamples.

## Summary

- Rules between labels (here, mutual exclusion) are clauses of the knowledge base, trained together with the labels.
- Any formula can be evaluated on the trained model, including formulas that are not in the knowledge base. Consequences of the knowledge base evaluate to true and contradictions to false; the value of any other formula depends on the data.
- A query can use its own, stricter operators: `with_updates()` copies the interpretation with the same groundings.